# What we tried since 25 September, measured one way

Since the last meeting (the prompt comparison), the work went in four directions:

1. **Better listener features.** The listener is the model of how the guesser
   picks words for a clue. New inputs: WordNet "is a kind of", ConceptNet
   relations, gpt-oss's free associations, and an association profile of the
   clue. A neural correction over raw embeddings was also tried, and so was
   association counts as an extra training target.
2. **Later picks of a turn.** The listener was trained mostly on how good a
   word is for the clue. After the first guess the guesser is less sure, and
   it leans toward words that go with what it already picked. Models of this:
   per-pick temperatures, the within-turn model, a joint refit, pick-index
   boosters, and history features.
3. **The decision rule.** The spymaster now chooses the clue that maximises
   the probability of winning after the turn, not expected words minus
   costs. Refinements of the value it uses were tried on top: turn
   calibration, a second round of policy iteration, a board-reading value, and
   a one-move lookahead at the opponent's reply.
4. **Learned clue policies.** A network maps the board straight to a clue. It
   was trained by imitation, then by REINFORCE on gpt-oss's reward, then by
   actor-critic on game results.

Everything is measured with **two numbers, kept apart on purpose**:

- **Listener accuracy** says how well a *model* predicts the guesser. It is
  McFadden R² on one fixed held-out set, reported by pick.
- **Win rate** says how well a *spymaster* plays. It depends on the model
  and also on how the spymaster uses the model's probabilities (the decision
  rule), so it is not a measure of the model.

**The same test for every row.** Every R² below is on the same 4,378 guesses.
Every win rate is on the same 100 boards, with the same guesser (gpt-oss-120b),
against the same opponent (the incumbent spymaster, unchanged since
September). Earlier results were spread over several sets and opponents, so
some numbers here differ a little from the ones in the log.

The tables are read from `cache/report/listener_accuracy.csv`
(`scripts/tools/report_listener_accuracy.py`) and `cache/report/win_rates.csv`
(`scripts/tools/report_win_rates.py`). Both are computed from cached models
and the stored games, and the notebook makes no API calls.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
acc = pd.read_csv(ROOT / "cache" / "report" / "listener_accuracy.csv").fillna({"base": ""})
wins = pd.read_csv(ROOT / "cache" / "report" / "win_rates.csv").fillna({"reference": ""})
EVAL, GPTOSS, TRAIN = "held-out words, generated, Sonnet 5.5", "held-out words, generated", "train"
PICKS = ["pick_1", "pick_2", "pick_3", "pick_4"]
pd.set_option("display.max_colwidth", 70)


def ci(x, lo, hi, signed=False):
    f = "{:+.4f}" if signed else "{:.4f}"
    return f"{f.format(x)} [{f.format(lo)}, {f.format(hi)}]"


def accuracy(group, base=None):
    # One row per model: R² on the eval set (pooled with its 95% CI, then by pick),
    # the paired gain over the incumbent, the training R² and the gpt-oss copy's R².
    a = acc[(acc.group == group) & ((acc.base == base) if base is not None else True)]
    e, t, g = (a[a.set == s].set_index(["base", "model"]) for s in (EVAL, TRAIN, GPTOSS))
    out = pd.DataFrame({
        "Sonnet 5.5 R² [95% CI]": [ci(r.r2, r.r2_lo, r.r2_hi) for r in e.itertuples()],
        **{f"pick {p[-1]}{'+' if p == 'pick_4' else ''}": e[p].map("{:.3f}".format).values for p in PICKS},
        "vs incumbent [95% CI]": [ci(r.vs_incumbent, r.vs_lo, r.vs_hi, True) for r in e.itertuples()],
        "training R²": t.loc[e.index, "r2"].map("{:.4f}".format).values,
        "gpt-oss R²": g.loc[e.index, "r2"].map("{:.4f}".format).values,
    }, index=[m for _, m in e.index])
    return out


def by_pick(group, base=None):
    # Training and eval R² side by side, by pick.
    a = acc[(acc.group == group) & ((acc.base == base) if base is not None else True)]
    e, t = (a[a.set == s].set_index(["base", "model"]) for s in (EVAL, TRAIN))
    cols = {}
    for p in ["r2"] + PICKS:
        name = "pooled" if p == "r2" else f"pick {p[-1]}{'+' if p == 'pick_4' else ''}"
        cols[name] = [f"{t.loc[i, p]:.3f} / {e.loc[i, p]:.3f}" for i in e.index]
    return pd.DataFrame(cols, index=[f"{b}: {m}" if group == "later picks" else m for b, m in e.index])


print(f"{acc[acc.set == EVAL].events.iloc[0]} eval events, {acc[acc.set == TRAIN].events.iloc[0]} training events; "
      f"{wins.boards.min()}-{wins.boards.max()} boards per challenger")

4378 eval events, 47223 training events; 88-99 boards per challenger


## 1. Listener accuracy

### The eval set

`held-out words, generated, Sonnet 5.5`:

- **Generated positions.** Boards, revealed words and clues are drawn exactly
  the way the training positions were drawn (`collect_listener_data.py`: the
  same clue mix, number and reveals). Positions from real games would be
  easier and skewed, because every clue in them was a spymaster's best choice.
- **Held-out words.** Every board uses only the 150 words that no model saw in
  training.
- **Ranked by Claude Sonnet 5.5**, not by gpt-oss. Every model was trained on
  gpt-oss's rankings, so a different guesser tests whether a gain is about
  guessing in general or about gpt-oss's habits. The last column is the same
  positions ranked by gpt-oss, for comparison.

It has 1,761 positions and **4,378 guesses**: 1,761 first picks, then 1,317,
866 and 434 at picks 2, 3 and 4+. It cost $5.36.

**McFadden R²** = 1 − (the model's log-loss on the guess actually made) / (the
log-loss of guessing uniformly among the words left). A score of 0 means no
better than uniform; a score of 1 means certain and always right. A guesser that is itself random
caps it below 1. For a rough sense, a guesser whose choices carry the
uncertainty Sonnet's do would cap it at about 0.67. The model guesses at
every pick from the words still on the board, so pick 1 chooses among about
20 words and pick 4 among about 17 (fewer at the end of a game).

**Training R²** is the same quantity on the 47,223 training guesses the
boosters were fitted on.

### 1a. Feature sets

Each row is one booster. Every pick is scored from the same scores over the
words left (the frozen model). Each feature row adds to the row above it
except the association-target row. That row is the incumbent's features
trained with association counts as an extra target.

In [2]:
accuracy("features")

,Sonnet 5.5 R² [95% CI],pick 1,pick 2,pick 3,pick 4+,vs incumbent [95% CI],training R²,gpt-oss R²
incumbent,"0.3205 [0.3031, 0.3383]",0.549,0.201,0.139,0.058,"+0.0000 [+0.0000, +0.0000]",0.4364,0.3186
44 recipe (train only),"0.3229 [0.3050, 0.3407]",0.548,0.206,0.143,0.064,"+0.0023 [-0.0005, +0.0052]",0.5121,0.3212
association counts as a target,"0.3230 [0.3051, 0.3405]",0.548,0.206,0.141,0.068,"+0.0024 [-0.0010, +0.0059]",0.4458,0.3196
+ is-a (WordNet),"0.3240 [0.3063, 0.3417]",0.549,0.207,0.145,0.062,"+0.0035 [+0.0004, +0.0066]",0.5135,0.3231
+ ConceptNet,"0.3240 [0.3054, 0.3421]",0.551,0.205,0.143,0.061,"+0.0035 [-0.0002, +0.0071]",0.5174,0.3235
+ free associations (assoc),"0.3293 [0.3107, 0.3473]",0.558,0.210,0.147,0.068,"+0.0088 [+0.0045, +0.0133]",0.5057,0.3340
+ association profile,"0.3309 [0.3125, 0.3492]",0.560,0.212,0.149,0.064,"+0.0104 [+0.0060, +0.0152]",0.5331,0.3404


**Reading.**

- **The association features are the only feature gain that clearly holds
  on Sonnet.**
  - Free associations add +0.0053 over ConceptNet (0.3240 → 0.3293).
  - The profile adds another +0.0016.
  - Together they put the eval R² at +0.0104 [+0.0060, +0.0152] over the
    incumbent.
  - Most of the gain is at pick 1 (0.548 → 0.560).
- **is-a and ConceptNet together are worth about +0.001** over the same
  recipe without them. These features were aimed at category clues, which are a
  small share of all clues.
- **Gains shrink when the guesser changes.** On the gpt-oss copy of the same
  positions, the two association steps add +0.0105 and +0.0064, about twice
  the Sonnet figures. Roughly half of what the association features learned
  is about guessing in general; the rest is about gpt-oss, whose own free
  associations they are.
- **Association counts as an extra training target** do not help here
  (0.3230, the same as the plain 44 recipe).
- **The incumbent is not the plain 44 recipe** (the second row). The incumbent
  has the same features but was fitted on train and val together, with 380
  trees. The 44 recipe was refitted on train only, with val used to stop, and
  has 2,590 trees. Every later booster uses that second recipe, so the
  difference between them (+0.0023, not significant) is the baseline for the
  feature gains.

### 1b. Later picks of a turn

Every model here keeps the base booster's first pick (up to small
differences). It changes how picks 2, 3 and 4+ are scored. There are two base
feature sets: the incumbent's 44 features, trained the same way as every
other row (`44`), and the 56 with ConceptNet, is-a and associations
(`assoc`).

| Model | What it does at pick j ≥ 2 | Playable? |
|---|---|---|
| frozen | softmax of the same scores over the words left | yes |
| + per-pick temperatures | scores × c_j, c_j fitted on val (ranking unchanged, less confident) | yes |
| fixed per-pick temperatures | the incumbent's scores ÷ 1.0 / 1.1 / 1.3 / 1.5 (pick_temperature_listener) | yes |
| + within-turn | α_j·score + β_j·fit. fit is the word's best similarity to a word already picked; α_j falls as the best word left gets worse than the best picked | yes |
| joint refit | the booster refitted with the within-turn layer inside its loss | yes |
| pick index | a booster trained on every pick, with the pick number j and j − k as inputs | yes |
| pick index + history | plus each word's similarity to the words already picked (embeddings, or all link sources) | not in play: about 130 booster runs per clue |

In [3]:
accuracy("later picks", "44")

,Sonnet 5.5 R² [95% CI],pick 1,pick 2,pick 3,pick 4+,vs incumbent [95% CI],training R²,gpt-oss R²
frozen,"0.3229 [0.3050, 0.3407]",0.548,0.206,0.143,0.064,"+0.0023 [-0.0005, +0.0052]",0.5121,0.3212
+ per-pick temperatures,"0.3295 [0.3129, 0.3458]",0.548,0.213,0.154,0.089,"+0.0090 [+0.0058, +0.0127]",0.4983,0.3298
+ within-turn,"0.3492 [0.3334, 0.3643]",0.548,0.241,0.192,0.131,"+0.0286 [+0.0229, +0.0351]",0.5108,0.3491
joint refit + within-turn,"0.3489 [0.3333, 0.3640]",0.546,0.243,0.191,0.133,"+0.0284 [+0.0224, +0.0348]",0.5332,0.3496
joint refit + per-pick temperatures,"0.3306 [0.3138, 0.3473]",0.543,0.218,0.160,0.094,"+0.0101 [+0.0063, +0.0138]",0.5035,0.3310
"pick index, depth k","0.3329 [0.3160, 0.3495]",0.549,0.220,0.161,0.085,"+0.0124 [+0.0084, +0.0169]",0.5209,0.3335
"pick index, depth 9","0.3331 [0.3156, 0.3498]",0.548,0.218,0.162,0.090,"+0.0126 [+0.0084, +0.0168]",0.5241,0.3356
"pick index + history, embeddings","0.3540 [0.3381, 0.3697]",0.549,0.245,0.203,0.142,"+0.0334 [+0.0273, +0.0402]",0.5509,0.3562
"pick index + history, all sources","0.3532 [0.3370, 0.3690]",0.548,0.244,0.203,0.142,"+0.0327 [+0.0264, +0.0396]",0.5630,0.3569


In [4]:
accuracy("later picks", "assoc")

,Sonnet 5.5 R² [95% CI],pick 1,pick 2,pick 3,pick 4+,vs incumbent [95% CI],training R²,gpt-oss R²
frozen,"0.3293 [0.3107, 0.3473]",0.558,0.210,0.147,0.068,"+0.0088 [+0.0045, +0.0133]",0.5057,0.3340
+ per-pick temperatures,"0.3350 [0.3178, 0.3516]",0.558,0.216,0.156,0.090,"+0.0145 [+0.0100, +0.0190]",0.4938,0.3413
+ within-turn,"0.3546 [0.3398, 0.3700]",0.558,0.244,0.194,0.133,"+0.0341 [+0.0278, +0.0408]",0.5058,0.3590
"pick index, depth 9","0.3374 [0.3196, 0.3545]",0.557,0.220,0.163,0.093,"+0.0169 [+0.0119, +0.0217]",0.5222,0.3447
"pick index + history, embeddings","0.3573 [0.3411, 0.3733]",0.557,0.245,0.203,0.142,"+0.0368 [+0.0302, +0.0435]",0.5667,0.3655
"pick index + history, all sources","0.3571 [0.3408, 0.3732]",0.556,0.246,0.202,0.144,"+0.0365 [+0.0299, +0.0434]",0.5737,0.3654


In [5]:
pd.concat([accuracy("later picks", "incumbent"), accuracy("later picks", "conceptnet")])

,Sonnet 5.5 R² [95% CI],pick 1,pick 2,pick 3,pick 4+,vs incumbent [95% CI],training R²,gpt-oss R²
fixed per-pick temperatures (pick_temperature_listener),"0.3271 [0.3107, 0.3424]",0.549,0.207,0.150,0.084,"+0.0065 [+0.0041, +0.0092]",0.4304,0.3274
within-turn (within_turn_listener),"0.3505 [0.3349, 0.3658]",0.551,0.241,0.191,0.132,"+0.0300 [+0.0239, +0.0369]",0.5152,0.3512


**Reading.**

- **Modelling later picks is the largest accuracy gain we have, about +0.03
  R².** Every feature set together gives about +0.01. Pick 1 does not move.
  The gain is at picks 2+: on the 44 base, pick 2 goes 0.206 → 0.245, pick 3
  0.143 → 0.204, and pick 4+ 0.064 → 0.142.
- **The order is the same on both bases.** From worst to best: frozen,
  temperatures, pick index, within-turn, pick index + history.
  - **Temperatures (+0.007)** only make the later picks less confident. They
    leave the ranking unchanged.
  - **Pick index (+0.010)** learns its own sharpness at each pick, and on val
    it is already almost calibrated. It does not know which words have been
    picked.
  - **Within-turn (+0.026)** adds that information: words that go with what
    was already picked become more likely. That is where most of the gain is.
  - **History (+0.003 to +0.005 over within-turn)** learns the same effect from the
    data, more flexibly. It is the most accurate model (0.3573 on the assoc
    base), but it costs about 130 booster runs per clue, so it cannot be used
    in play yet.
- **Features and the later-pick model add up.** The assoc base is better than
  the 44 base by about +0.003 to +0.006 under every later-pick model.
- **The joint refit adds nothing** over fitting the booster first and the
  within-turn layer after (0.3489 against 0.3492).
- **The two rows used in play** match their counterparts above. The fixed
  pick temperatures are +0.0065 on the incumbent, and the deployed
  within-turn model on the ConceptNet booster scores 0.3505.

### 1c. Neural correction

A small network over raw word embeddings is trained to correct the 44-feature
booster's scores (learned similarity heads, no attention), with three seeds.
On training positions it adds to cross-fitted booster scores, as it was trained.

In [6]:
accuracy("neural")

,Sonnet 5.5 R² [95% CI],pick 1,pick 2,pick 3,pick 4+,vs incumbent [95% CI],training R²,gpt-oss R²
44 recipe + neural correction (seed 0),"0.3273 [0.3083, 0.3453]",0.549,0.209,0.155,0.069,"+0.0067 [+0.0018, +0.0118]",0.4130,0.3352
44 recipe + neural correction (seed 1),"0.3222 [0.3025, 0.3408]",0.548,0.200,0.150,0.060,"+0.0017 [-0.0036, +0.0076]",0.4351,0.3352
44 recipe + neural correction (seed 2),"0.3262 [0.3068, 0.3451]",0.550,0.208,0.152,0.065,"+0.0057 [+0.0006, +0.0113]",0.4256,0.3346


**Reading.**

- **The neural correction is a gain on gpt-oss that mostly does not carry
  over to Sonnet.** Over its own base (the 44 recipe), it adds +0.013 to
  +0.014 R² on gpt-oss's copy of the eval set. On Sonnet 5.5 it adds
  −0.001 to +0.004, depending on the seed.
- It learned from raw embeddings what gpt-oss in particular does with words.
  It was not developed further, and no spymaster uses it.

### 1d. Training against eval, by pick

Each cell is training R² / eval (Sonnet 5.5) R².

In [7]:
pd.concat([by_pick("features"), by_pick("later picks", "assoc"), by_pick("neural")])

,pooled,pick 1,pick 2,pick 3,pick 4+
incumbent,0.436 / 0.321,0.657 / 0.549,0.357 / 0.201,0.222 / 0.139,0.148 / 0.058
44 recipe (train only),0.512 / 0.323,0.717 / 0.548,0.449 / 0.206,0.308 / 0.143,0.221 / 0.064
association counts as a target,0.446 / 0.323,0.661 / 0.548,0.371 / 0.206,0.236 / 0.141,0.159 / 0.068
+ is-a (WordNet),0.514 / 0.324,0.718 / 0.549,0.451 / 0.207,0.310 / 0.145,0.222 / 0.062
+ ConceptNet,0.517 / 0.324,0.722 / 0.551,0.455 / 0.205,0.313 / 0.143,0.225 / 0.061
+ free associations (assoc),0.506 / 0.329,0.716 / 0.558,0.440 / 0.210,0.296 / 0.147,0.212 / 0.068
+ association profile,0.533 / 0.331,0.739 / 0.560,0.472 / 0.212,0.327 / 0.149,0.236 / 0.064
assoc: frozen,0.506 / 0.329,0.716 / 0.558,0.440 / 0.210,0.296 / 0.147,0.212 / 0.068
assoc: + per-pick temperatures,0.494 / 0.335,0.716 / 0.558,0.426 / 0.216,0.269 / 0.156,0.184 / 0.090
assoc: + within-turn,0.506 / 0.355,0.716 / 0.558,0.445 / 0.244,0.291 / 0.194,0.206 / 0.133


**Reading.**

- **Training R² runs 0.50–0.57; eval R² runs 0.32–0.36.**
- **The gap can be split into steps, using the assoc booster:**

  | Positions | Guesser | R² |
  |---|---|---|
  | its training positions | gpt-oss | 0.506 |
  | val: new boards, training words | gpt-oss | 0.371 |
  | held-out words | gpt-oss | 0.334 |
  | held-out words | Sonnet 5.5 | 0.329 |

  Most of the gap is in-sample fit, then new words. A different guesser costs
  little.
- **In-sample fit depends on the number of trees and says little about
  quality.** The incumbent (380 trees) scores 0.436 on training data, and the
  44 recipe (2,590 trees) 0.512. Their eval R² is nearly the same.
- **Later picks have the largest gap.** Pick 4+ has 0.22 on training data for
  the frozen booster against 0.06 on eval. The booster is confident at later
  picks on data it has seen, and wrong on data it has not. This is what the
  later-pick models fix.
- **The neural rows' training R² is not comparable to the others.** It is
  computed on top of cross-fitted booster scores, which are out-of-sample.

## 2. Win rates

**The test.** The gpt-oss suite has 100 fixed boards, and each is played
twice with the seats swapped. gpt-oss-120b guesses for both sides. The
opponent is always the incumbent (`learned_listener`: expected words minus
role costs, under the September booster). Each row is one model, looked up by
a hash of its parameters and model files.

**Columns.**
- **win%** is over the games on the boards the challenger played both ways
  (88–99; a board drops out when gpt-oss returned no usable ranking).
- **swept** is boards the challenger won both ways against boards the
  incumbent won both ways. **sign p** is the exact sign test on those; this is
  the paired test.
- **common** is the win% on the 49 boards that every challenger played both
  ways. It checks that the order is not an accident of which boards each row
  kept.
- **vs reference** is used where a row changes one thing relative to another
  row: one booster under a fixed decision rule, or one decision rule with a
  fixed booster. It is the paired difference in win rate over the boards
  both played, with a 95% bootstrap over boards.

In [8]:
def win_table(group):
    w = wins[wins.group == group]
    return pd.DataFrame({
        "boards": w.boards.values,
        "win% [95% CI]": [f"{r.win:.1%} [{r.win_lo:.2f}, {r.win_hi:.2f}]" for r in w.itertuples()],
        "swept": [f"{r.swept_for} vs {r.swept_against}" for r in w.itertuples()],
        "sign p": w.sign_p.map("{:.3f}".format).values,
        "common (49)": w.common_win.map("{:.1%}".format).values,
        "assassin": [f"{r.assassin} vs {r.assassin_incumbent}" for r in w.itertuples()],
        "mean k": w.mean_k.map("{:.2f}".format).values,
        "vs reference [95% CI]": ["" if not r.reference else
                                  f"{r.vs_ref:+.1%} [{r.vs_lo:+.1%}, {r.vs_hi:+.1%}] vs {r.reference}"
                                  for r in w.itertuples()],
    }, index=w.model.values)

### 2a. Better listeners, the incumbent's decision rule

These are the incumbent's spymaster with a new booster swapped in.

In [9]:
win_table("listener, incumbent's objective")

,boards,win% [95% CI],swept,sign p,common (49),assassin,mean k,vs reference [95% CI]
is-a features (isa_listener),94,"52.1% [0.45, 0.59]",19 vs 15,0.608,46.9%,13 vs 10,2.18,
+ ConceptNet (conceptnet_listener),96,"54.7% [0.48, 0.62]",24 vs 15,0.200,55.1%,11 vs 10,2.19,
+ free associations (assoc_feature_listener),96,"53.1% [0.46, 0.60]",23 vs 17,0.430,53.1%,10 vs 5,2.19,
"association-count target, no pass",99,"48.0% [0.41, 0.55]",16 vs 20,0.618,45.9%,10 vs 14,2.20,
association-count target + pass at 5%,97,"41.8% [0.35, 0.49]",12 vs 28,0.017,44.9%,6 vs 13,1.89,"-6.2% [-14.6%, +2.6%] vs association-count target, no pass"


**Reading.**

- **No booster is a significant gain under the old decision rule.**
  ConceptNet is the best at 54.7% (24 vs 15 boards, p = 0.20).
- **The pass loses in this arena.** The association-count booster plays even
  without the pass (48.0%). With the pass it loses: 41.8%, 12 vs 28 boards,
  p = 0.017.
  - The pass makes the spymaster give fewer words (mean k 1.89), and gpt-oss
    does not reward that, because it keeps guessing even when it has nothing
    good.
  - A human guesser would stop, so this arena probably understates what the
    pass is worth against people.

### 2b. Better listeners, the win-probability decision rule

`win_prob_listener` scores a clue by the probability of winning after the turn.
Each way the turn can end (from the listener's probabilities) leaves a score
of a own words left and b opponent words left. That score is valued by V(a, b),
the incumbent's win rate from that score in 10,266 recorded gpt-oss games.
There are no hand-set costs. The booster is the only thing that changes down
this table.

In [10]:
win_table("listener, win objective")

,boards,win% [95% CI],swept,sign p,common (49),assassin,mean k,vs reference [95% CI]
"win_prob, incumbent booster",98,"51.5% [0.45, 0.58]",17 vs 14,0.720,55.1%,31 vs 9,2.68,
"win_prob, is-a booster",96,"53.1% [0.46, 0.60]",22 vs 16,0.418,56.1%,31 vs 7,2.63,"+2.1% [-4.2%, +8.9%] vs win_prob, incumbent booster"
"win_prob, ConceptNet booster",93,"57.5% [0.50, 0.64]",24 vs 10,0.024,56.1%,25 vs 11,2.62,"+5.5% [-2.2%, +13.2%] vs win_prob, incumbent booster"
"win_prob, assoc booster",99,"59.1% [0.52, 0.66]",28 vs 10,0.005,57.1%,21 vs 10,2.59,"+7.7% [+0.5%, +14.8%] vs win_prob, incumbent booster"
"win_prob, assoc profile booster",90,"57.2% [0.50, 0.64]",20 vs 7,0.019,57.1%,18 vs 9,2.68,"+5.7% [-1.1%, +12.5%] vs win_prob, incumbent booster"


**Reading.**

- **The decision rule is where the wins come from.** With the incumbent's own
  booster, the win-probability rule is even with the incumbent (51.5%).
- **Better boosters are worth more under it than under the old rule.** The
  assoc booster wins 59.1% (28 vs 10 boards, p = 0.005), against 53.1% for
  the same booster under the old rule.
  - Its paired gain over the incumbent booster under the same rule is +7.7
    points [+0.5, +14.8]. That is the only booster-against-booster
    comparison in this notebook whose interval excludes zero.
- **Win rate roughly follows R² down this table**, but the order is noisy.
  The booster with the highest R², the association profile, plays like assoc.
- **The rule raises the number given**, from about 2.2 to about 2.6. It also
  hits the assassin more: 18 to 31 assassin losses against the incumbent's
  7 to 11.

### 2c. Later-pick models in play

In [11]:
win_table("later picks")

,boards,win% [95% CI],swept,sign p,common (49),assassin,mean k,vs reference [95% CI]
per-pick temperatures (pick_temperature_listener),97,"44.3% [0.38, 0.51]",8 vs 19,0.052,39.8%,7 vs 9,1.92,
"within-turn, incumbent's objective (within_turn_listener)",95,"41.6% [0.35, 0.49]",11 vs 27,0.014,41.8%,5 vs 6,2.00,
"within-turn, fit term only",98,"49.0% [0.42, 0.56]",17 vs 19,0.868,46.9%,10 vs 9,2.19,"+6.4% [-1.1%, +13.8%] vs within-turn, incumbent's objective (withi..."
"win_prob, ConceptNet booster + within-turn",96,"55.7% [0.49, 0.63]",22 vs 11,0.080,59.2%,30 vs 9,2.43,"+0.0% [-7.3%, +7.9%] vs win_prob, ConceptNet booster"


**Reading.**

- **The later-pick models that gained R² lost games under the old rule.**
  - Fixed pick temperatures: 44.3%, 8 vs 19 boards, p = 0.052.
  - Within-turn: 41.6%, 11 vs 27 boards, p = 0.014.
- **Both lowered the number given**, from 2.2 to 1.9–2.0. A model that is
  honest about later picks makes multi-word clues look worth less. Under the
  old rule's fixed costs, the spymaster then gives smaller clues and finds
  fewer words per turn.
- **The fit term alone** (the within-turn model without the change in
  confidence) is even at 49.0%.
- **Under the win-probability rule, within-turn makes no difference**: +0.0
  points [−7.3, +7.9] against the same booster with the frozen model.

### 2d. Refinements of the decision rule

- **Turn calibration** corrects the listener's probabilities on the turns a
  spymaster actually chose. Those turns are selected for looking good, which
  is the optimiser's curse.
- **V2** re-estimates V from games where win_prob itself plays, as one round
  of policy iteration.
- **Board-reading V** adds a learned board correction to V(a, b).
- **Reply lookahead** works out the incumbent's reply on each board our clue
  could leave.

In [12]:
win_table("decision rule")

,boards,win% [95% CI],swept,sign p,common (49),assassin,mean k,vs reference [95% CI]
"win_prob, assoc booster + turn calibration",97,"54.1% [0.47, 0.61]",22 vs 14,0.243,57.1%,24 vs 6,2.39,"-5.2% [-10.8%, +0.5%] vs win_prob, assoc booster"
"win_prob, assoc booster, V2 (policy iteration)",96,"52.6% [0.46, 0.60]",19 vs 14,0.487,56.1%,23 vs 11,2.54,"-5.8% [-12.6%, +1.1%] vs win_prob, assoc booster"
board-reading V (board_value_listener),88,"51.1% [0.44, 0.58]",13 vs 11,0.839,48.0%,35 vs 9,2.59,"-1.1% [-8.0%, +6.3%] vs win_prob, incumbent booster"
reply lookahead (reply_lookahead_listener),92,"56.0% [0.49, 0.63]",20 vs 9,0.061,57.1%,28 vs 6,2.59,"+3.9% [-2.2%, +10.0%] vs win_prob, incumbent booster"


**Reading.**

- **No refinement of the rule is a significant change.**
- **Calibration and V2 both lean the wrong way**, by about −5 points.
  Calibration lowers the number given (2.39). The intervals just include
  zero.
- **The board-reading V** is even with its reference.
- **The reply lookahead is the best of these**: 56.0%, 20 vs 9 boards,
  p = 0.061. It is +3.9 points [−2.2, +10.0] over the same rule without it.
  On 600 simulated boards it was +2.7 [−0.1, +5.4], in the same direction.

### 2e. Learned clue policies

In [13]:
win_table("clue policy")

,boards,win% [95% CI],swept,sign p,common (49),assassin,mean k,vs reference [95% CI]
imitation of the incumbent (imitation_policy),89,"38.8% [0.32, 0.46]",6 vs 26,0.001,37.8%,16 vs 6,2.31,
+ REINFORCE on gpt-oss reward (gptoss_reward_policy),89,"43.8% [0.37, 0.51]",10 vs 21,0.071,41.8%,15 vs 5,2.08,"+4.9% [-1.8%, +11.6%] vs imitation of the incumbent (imitation_pol..."
actor-critic on game results (win_actor_critic),90,"46.7% [0.40, 0.54]",17 vs 23,0.430,46.9%,24 vs 6,2.50,


**Reading.**

- **No learned policy matches the incumbent.**
- **Imitation loses badly**: 38.8%, 6 vs 26 boards. It picks the incumbent's
  clue about half the time, and its other clues are worse.
- **REINFORCE on gpt-oss's reward** recovers +4.9 points [−1.8, +11.6] over
  imitation.
- **Actor-critic on game results** (no listener inside) comes closest: 46.7%,
  17 vs 23 boards. It gives bigger clues (2.50) and hits the assassin 24
  times against 6.

## 3. The two measures together

- **Accuracy and wins agree on boosters when the decision rule uses the
  probabilities.**
  - Under the win-probability rule, the boosters with higher R² mostly win
    more.
  - Under the old rule, the probabilities are offset by hand-set costs, and
    the same R² gains do not show up as wins.
- **The largest accuracy gain has not yet produced wins.**
  - The later-pick models are about three times the feature gains in R².
  - Under the old rule they lost games by lowering the number given.
  - Under the win-probability rule the within-turn model made no
    difference.
  - The most accurate model, pick index + history, has not been played at
    all, because it is too slow.
  - So whether a more accurate model of later picks helps play is still
    open. A spymaster built to use it would need to evaluate it cheaply
    enough to search over clues.
- **Rule refinements leave R² unchanged by construction** (the model is the
  same). Their win-rate changes are all within noise. The reply lookahead is
  the only one pointing up.
- **Best spymaster so far**: the win-probability rule with the assoc booster.
  It wins 59.1% on its own boards and 57.1% on the 49 common boards. The
  common-board column keeps the same broad order: the win-probability rows
  are at the top, then the old-rule boosters, then the later-pick models and
  the policies.
- **Best listener so far**: the assoc booster with pick index + history,
  0.3573 on Sonnet 5.5.
- **Next step**: put a cheap version of that later-pick model (within-turn is
  playable today) under the win-probability rule with the assoc booster.
  Measure accuracy on this eval set and play on this suite.

## 4. Changes to the evaluation itself

- **A new eval set**: the Sonnet 5.5 copy described in section 1. An earlier
  Sonnet 5 copy (889 positions, $2.85) is no longer used, for the reason
  below.
- **Unparsed LLM answers were stored as board order.** When an Anthropic
  answer could not be parsed, the guesser fell back to listing the words in
  board order and stored that as if it were a ranking. This happened when an
  answer was cut off at the 512-token cap, or when the model corrected itself
  and wrote several arrays. It affected 8.5% of the Sonnet 5 copy, which is
  why that copy is dropped.

  Now the guesser asks again, up to 3 times. If it still has no usable
  answer, it raises an error and stores nothing. The cap is 2,048 tokens, and
  the last valid array in the answer is the one used.
- **The win-rate test is one test for every spymaster.** It is the same suite,
  guesser and opponent for all of them. The three clue policies were played on
  it for this notebook ($0.16 of DeepInfra credit), so all 21 rows are
  comparable.

## Caveats

- **About 90 boards per spymaster.**
  - A 95% interval on a single win rate is about ±7 points.
  - With 21 rows, one or two sign tests at p < 0.05 are expected by chance.
    The p-values are not corrected for this.
  - Paired differences between rows are tighter, but most of them still
    cross zero.
- **gpt-oss is both the guesser the listener was trained on and the guesser
  in every game.** These win rates are against gpt-oss. People pass when
  unsure, and gpt-oss keeps guessing, so rules that give small or cautious
  clues may look worse here than they would against people.
- **Boards drop out when gpt-oss returns no usable ranking** (about 10 per
  challenger). Each row therefore has a slightly different set of boards,
  which is why the common-boards column is there.
- **The incumbent booster was fitted on train and val**, with a different
  number of trees, so its training R² is not comparable to the other
  boosters'. Its eval R² is.
- **The neural rows' training R² uses cross-fitted base scores**, as they
  were trained.
- **R² intervals are a bootstrap over boards.** Every eval position is on its
  own board, so this is a bootstrap over positions.
- **The 0.67 ceiling is an illustration.** It is the R² of a perfect model if
  Sonnet's choices were as spread out as an assumed distribution. It was not
  measured.
- **About 1.5–2.3% of the gpt-oss training rankings** were backfilled the
  same way as the Sonnet copy (found 2026-09-29). They are in every model's
  training data equally.